This file was purely generated using AI - just to get all the features of a particular cutomer. This was helping in testing.
Note - In real prod environments I would not keep this, but since i used this method, I decided to let it stay in my submission just for reference.

In [2]:
import pandas as pd
import json

def get_customer_360(user_id: str, data_dir: str = 'data/') -> dict:
    """
    Fetches a comprehensive 360-degree view of a customer by aggregating 
    network logs, subscriptions, plan details, and support transcripts.
    """
    try:
        # Load all datasets
        logs_df = pd.read_csv(f'{data_dir}tower_logs.csv')
        subs_df = pd.read_csv(f'{data_dir}customer_subscriptions.csv')
        plans_df = pd.read_csv(f'{data_dir}plans.csv')
        transcripts_df = pd.read_csv(f'{data_dir}transcripts.csv')
    except FileNotFoundError as e:
        return {"error": f"Data file not found: {e}"}

    # Extract user-specific network logs
    user_logs = logs_df[logs_df['user_id'] == user_id]
    if user_logs.empty:
        return {"error": f"Customer {user_id} not found in tower logs."}
    
    # Extract subscription and join with plan features
    user_sub = subs_df[subs_df['user_id'] == user_id]
    if not user_sub.empty:
        plan_id = user_sub.iloc[0]['current_plan_id']
        plan_details = plans_df[plans_df['plan_id'] == plan_id].iloc[0]
    else:
        plan_id = "Unknown"
        plan_details = pd.Series({"plan_name": "None", "data_limit_gb": 0, "price_usd": 0})

    # Extract all transcripts
    user_transcripts = transcripts_df[transcripts_df['user_id'] == user_id]

    # Construct the final 360 dictionary
    customer_profile = {
        "user_id": user_id,
        "churn_risk_label": int(user_logs.iloc[0]['label']),
        "network_metrics": {
            "signal_strength": round(float(user_logs.iloc[0]['signal_strength']), 2),
            "latency_ms": round(float(user_logs.iloc[0]['latency']), 2),
            "dropped_calls": int(user_logs.iloc[0]['dropped_calls']),
            "avg_data_gb": round(float(user_logs.iloc[0]['avg_data_gb']), 2)
        },
        "subscription_details": {
            "current_plan_id": plan_id,
            "plan_name": plan_details['plan_name'],
            "data_limit_gb": int(plan_details['data_limit_gb']),
            "monthly_price_usd": float(plan_details['price_usd'])
        },
        "support_history": user_transcripts['text_transcript'].tolist()
    }
    
    return customer_profile

In [3]:

target_customer = "CUST_376"

print(f"--- Fetching 360 View for {target_customer} ---")
profile = get_customer_360(target_customer)

# Print as beautifully formatted JSON
print(json.dumps(profile, indent=4))

--- Fetching 360 View for CUST_376 ---
{
    "user_id": "CUST_376",
    "churn_risk_label": 0,
    "network_metrics": {
        "signal_strength": -67.76,
        "latency_ms": 38.46,
        "dropped_calls": 1,
        "avg_data_gb": 72.02
    },
    "subscription_details": {
        "current_plan_id": "P_STANDARD",
        "plan_name": "Standard 50GB",
        "data_limit_gb": 50,
        "monthly_price_usd": 30.0
    },
    "support_history": [
        "Customer says: The service is okay and question. I want a question.",
        "Customer says: The service is update and okay. I want a question."
    ]
}
